In [ ]:
###LOAD NMR DATA

import os
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import numpy as np
import nmrglue as ng

def find_experiment_number(subdirectory, experiment_type):
    """
    Searches for the NOESY or CPMG experiment number in the subdirectories of a Bruker dataset.

    Parameters
    ----------
    subdirectory : str
        Directory path where the experiments are located.
    experiment_type : str
        Type of experiment ('noesy' or 'cpmg').

    Returns
    -------
    str or None
        Experiment number as a string, or None if not found.
    """
    exp_marker = "noesygppr1d" if experiment_type == "noesy" else "cpmgpr1d"
    
    for item in sorted(os.listdir(subdirectory)):
        if item.isdigit() and os.path.isdir(os.path.join(subdirectory, item)):
            pulseprogram_path = os.path.join(subdirectory, item, "pulseprogram")
            if os.path.exists(pulseprogram_path):
                with open(pulseprogram_path, 'r') as file:
                    if exp_marker in file.read():
                        return item
    return None

def baseline_asls(y, lam=1e4, p=0.01, n_iter=10):
    """
    Baseline correction using Asymmetric Least Squares (AsLS) smoothing.

    Parameters
    ----------
    y : numpy.ndarray
        Input signal array (intensity values of the spectrum region).
    lam : float, optional
        Smoothness parameter (lambda). Default is 1e4.
    p : float, optional
        Asymmetry parameter (penalizes positive residuals more than negative). Default is 0.01.
    n_iter : int, optional
        Number of iterations for the optimization loop. Default is 10.

    Returns
    -------
    numpy.ndarray
        Estimated baseline array of the same length as the input signal.
    """
    L = len(y)
    D = np.zeros((L, L))
    
    # Construct the difference matrix
    for i in range(L - 2):
        D[i, i] = 1
        D[i, i + 1] = -2
        D[i, i + 2] = 1
        
    D = lam * D.T @ D
    w = np.ones(L)
    
    # Iterative baseline estimation
    for _ in range(n_iter):
        W = np.diag(w)
        Z = np.linalg.solve(W + D, w * y)
        w = p * (y > Z) + (1 - p) * (y < Z)
        
    return Z

def calibrate_spectrum(ppms, spectrum, alanine_range=(1.48, 1.52), target_ppm=1.496, ppm_range=(10.95, -0.95)):
    """
    Calibrates an NMR spectrum using the alanine doublet peaks and then truncates it to the desired ppm range.
    
    Parameters
    ----------
    ppms : numpy.ndarray
        Array of ppm values.
    spectrum : numpy.ndarray
        Array of intensity values for the spectrum.
    alanine_range : tuple of float, optional
        The ppm range to search for the alanine peaks. Default is (1.48, 1.52).
    target_ppm : float, optional
        The exact theoretical ppm value for the center of the alanine doublet. Default is 1.496.
    ppm_range : tuple of float, optional
        The final ppm range to keep after calibration. Default is (10.95, -0.95).
        
    Returns
    -------
    tuple
        A tuple containing the truncated ppm array and the calibrated, truncated spectrum array.
    """
    # --- Alanine calibration phase ---
    # Alanine range for calibration (uses the full spectrum)
    alanine_mask = (ppms >= alanine_range[0]) & (ppms <= alanine_range[1])
    if not np.any(alanine_mask):
        # If alanine is not found, return the spectrum unchanged
        return ppms, spectrum  
    
    data_alanine = spectrum[alanine_mask]
    
    # Baseline correction for the alanine region
    baseline = baseline_asls(data_alanine, lam=1e5, p=0.01, n_iter=10)
    data_alanine_corrected = data_alanine - baseline
    
    # Find peaks in the corrected alanine signal
    peak_indices, _ = find_peaks(data_alanine_corrected, height=np.max(data_alanine_corrected) * 0.5)
    
    if len(peak_indices) == 2:
        # Find the central position of the alanine peaks
        alanine_peaks = ppms[alanine_mask][peak_indices]
        alanine_center = np.mean(alanine_peaks)
        calibration_shift = target_ppm - alanine_center

        # Apply the shift to the entire ppm scale
        shifted_ppms = ppms + calibration_shift
        interpolator = interp1d(shifted_ppms, spectrum, bounds_error=False, fill_value=0)
        calibrated_spectrum = interpolator(ppms)
    else:
        # If the two alanine peaks are not detected, return the spectrum unchanged
        calibrated_spectrum = spectrum  

    # --- Truncation phase ---
    # Apply the truncation after calibration
    ppm_mask = (ppms >= ppm_range[1]) & (ppms <= ppm_range[0])
    ppms_cut = ppms[ppm_mask]
    spectrum_cut = calibrated_spectrum[ppm_mask]

    return ppms_cut, spectrum_cut

def load_nmr_spectra(data_path, output_file, ref_ppm_file, ref_file='', cpmg=False, is_urine=False, is_80mhz=False, extract_fid=False):
    """
    Loads and saves NMR spectra into a DataFrame and CSV file from raw Bruker data.
    Automatically detects the experiment number and calibrates using the alanine signal.

    Outputs generated:
    - The main processed dataset is saved exactly as named in `output_file`.
    - If `extract_fid` is True, three additional truncated files (first 16384 points) 
      are automatically saved by appending suffixes to the `output_file` name:
        * '_tr16384_real.csv' (Real part only)
        * '_tr16384_imag.csv' (Imaginary part only)
        * '_tr16384_ir.csv'   (Both real and imaginary parts concatenated)

    Parameters
    ----------
    data_path : str
        Path to the Bruker data directory.
    output_file : str
        Name and path of the main output CSV file.
    ref_ppm_file : str
        Path to the CSV file containing the reference PPM vector (e.g., 'dataPPMS_NOESY.csv').
    ref_file : str, optional
        Path to a reference CSV containing previously loaded spectra to append to. Default is empty.
    cpmg : bool, optional
        Set to True to select CPMG experiment instead of NOESY. Default is False.
    is_urine : bool, optional
        Set to True if the sample is urine (ERETIC signal at 12 ppm). Default is False.
    is_80mhz : bool, optional
        Set to True if the sample is from an 80 MHz spectrometer (no ERETIC signal). Default is False.
    extract_fid : bool, optional
        Set to True to extract and process the Free Induction Decay (FID). Default is False.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the loaded and calibrated spectra.
    """
    
    # --- Initialization and reference checking ---
    # Check if a reference spectra file is provided and exists
    has_ref_file = os.path.isfile(ref_file)
    if has_ref_file:
        ref_df = pd.read_csv(ref_file, index_col=0)
        if not extract_fid:
            ref_df = ref_df.rename(columns=lambda x: float(x))
        ref_indices = list(ref_df.index)

    # --- Directory traversal and experiment detection ---
    # Find all sample directories containing an 'nmr' subfolder
    sample_paths = [f2.path for f in os.scandir(data_path) if f.is_dir() for f2 in os.scandir(os.path.join(f.path, 'nmr')) if f2.is_dir()]
    
    # Automatically search for the correct experiment number
    example_path = sample_paths[0] if sample_paths else None
    if example_path:
        exp_num = find_experiment_number(example_path, "cpmg" if cpmg else "noesy")
        if not exp_num:
            raise FileNotFoundError(f"Experiment number for {'CPMG' if cpmg else 'NOESY'} not found in {example_path}")
    else:
        raise FileNotFoundError("No spectrum directories found in the provided path.")

    # Load the reference ppm scale explicitly from the provided file
    if not os.path.isfile(ref_ppm_file):
        raise FileNotFoundError(f"Reference PPM file not found: {ref_ppm_file}")
    ref_ppms = pd.read_csv(ref_ppm_file, header=None).to_numpy().squeeze()
    
    valid_sample_paths = sample_paths[:]
    count = 0
    is_first_iteration = True

    # --- Main sample processing loop ---
    for path in sample_paths:
        count += 1
        print(f"{count} of {len(sample_paths)}", end="\r")
        
        try:
            sample_name = path.split('/')[-1]
            if has_ref_file and sample_name in ref_indices:
                valid_sample_paths.remove(path)
                continue
            
            path_spectrum = os.path.join(path, exp_num, 'pdata', '1', '1r')
            
            if os.path.isfile(path_spectrum):
                # --- FID Extraction mode ---
                if extract_fid:
                    meta_dict, spectrum_data = ng.bruker.read(os.path.join(path, exp_num))
                    df = pd.DataFrame({'real': spectrum_data.real, 'imag': spectrum_data.imag})
                    
                    index_suffix = 'j'
                    df_real = df['real'].reset_index(drop=True)
                    df_real.index = df_real.index.astype(str)
                    
                    df_imag = df['imag'].reset_index(drop=True)
                    df_imag.index = df_imag.index.astype(str) + index_suffix
                    
                    new_df = pd.concat([df_real, df_imag])
                    
                    if is_first_iteration:
                        matrix = new_df.index.to_numpy().squeeze()
                        is_first_iteration = False
                    
                    new_df = new_df / new_df.max()
                    matrix = np.vstack([matrix, new_df.to_numpy().squeeze()])
                    
                # --- Spectrum Processing mode ---
                else:
                    meta_dict, spectrum_data = ng.bruker.read_pdata(os.path.join(path, exp_num, 'pdata', '1'))
                    
                    start_ppm = float(meta_dict['procs']['OFFSET'])
                    end_ppm = start_ppm - float(meta_dict['acqus']['SW'])
                    ftsize = meta_dict['procs']['FTSIZE']
                    step = float(meta_dict['acqus']['SW']) / ftsize
                    ppms = np.arange(start_ppm, end_ppm, -step)[:ftsize]
                    
                    spectrum_data = np.flip(np.interp(np.flip(ref_ppms), np.flip(ppms), np.flip(spectrum_data)))
        
                    # --- ERETIC/Reference normalization ---
                    # Indices 33600:33700 (urine) and 20570:20572 (serum) are hardcoded 
                    # to match the specific ppm position of the reference signal. 
                    if is_urine:
                        norm_int = spectrum_data[33600:33700].max() / 10000
                    else:
                        norm_int = spectrum_data[20570:20572].max() / 10000
        
                    # --- CPMG specific processing ---
                    if cpmg: 
                        noesy_exp_num = find_experiment_number(example_path, "noesy")
                        noesy_path = os.path.join(path, noesy_exp_num, 'pdata', '1')
                        
                        if os.path.isfile(os.path.join(noesy_path, '1r')):
                            meta_dict2, spectrum_data2 = ng.bruker.read_pdata(noesy_path)
                            start_ppm2 = float(meta_dict2['procs']['OFFSET'])
                            end_ppm2 = start_ppm2 - float(meta_dict2['acqus']['SW'])
                            ftsize2 = meta_dict2['procs']['FTSIZE']
                            step2 = float(meta_dict2['acqus']['SW']) / ftsize2
                            ppms2 = np.arange(start_ppm2, end_ppm2, -step2)[:ftsize2]
                            
                            spectrum_data2 = np.flip(np.interp(np.flip(ref_ppms), np.flip(ppms2), np.flip(spectrum_data2)))
                            
                            # Same hardcoded indices requirement applies here
                            if is_urine:
                                norm_int = spectrum_data2[33600:33700].max() / 10000
                            else:
                                norm_int = spectrum_data2[20570:20572].max() / 10000
                                
                    if not is_80mhz:
                        spectrum_data /= norm_int
        
                    # --- Calibration and matrix assembly ---
                    ppms_cal, spectrum_data = np.round_(calibrate_spectrum(ref_ppms, spectrum_data), decimals=5)
                    
                    if is_first_iteration:
                        matrix = ppms_cal[:]
                        is_first_iteration = False
                        
                    matrix = np.vstack([matrix, spectrum_data])
            else:
                valid_sample_paths.remove(path)
                
        except ValueError as e:
            print(f"\n\n#################################################")
            print(f" ERROR OCCURRED IN FOLDER: {path}")
            print(f"#################################################\n")
            raise e

    # --- DataFrame creation and export ---
    sample_names_clean = [p.split('/')[-1] for p in valid_sample_paths]
    spectra_df = pd.DataFrame(data=matrix[1:], index=sample_names_clean, columns=matrix[0])

    if has_ref_file:
        spectra_df = pd.concat([ref_df, spectra_df])

    spectra_df.to_csv(output_file)
    
    # --- Truncated FID export ---
    if extract_fid:
        print('Original shape:', spectra_df.shape)
        print('Separating real and imaginary...')
        
        data1 = spectra_df.filter(regex='j')
        print('Imaginary shape:', data1.shape)
        
        data0 = spectra_df[spectra_df.columns.drop(list(spectra_df.filter(regex='j')))]
        print('Real shape:', data0.shape)
        
        print('Truncating...')
        data0 = data0.iloc[:, :16384]
        print('Real shape (truncated):', data0.shape)
        
        data1 = data1.iloc[:, :16384]
        print('Imaginary shape (truncated):', data1.shape)
        
        print('Saving real truncated...')
        data0.to_csv(output_file.replace('.csv', '_tr16384_real.csv'))
        
        print('Saving imaginary truncated...')
        data1.to_csv(output_file.replace('.csv', '_tr16384_imag.csv'))
        
        print('Joining and saving imaginary+real truncated...')
        pd.concat([data0, data1], axis=1).to_csv(output_file.replace('.csv', '_tr16384_ir.csv'))

    return spectra_df

In [ ]:
###NMR DATA BINNING

import pandas as pd
import numpy as np
import re

def bin_spectra(file_path, bin_sizes, decimals=5):
    """
    Bins NMR spectra data across multiple specified bin sizes.
    
    - Groups consecutive data points by position (without interpolation).
    - Includes the final bin even if it contains fewer points than the specified bin size.
    - Input columns must be numeric (supports scientific notation, e.g., '-9e-05').
    - Output column labels are set to the mathematical mean of the grouped ppm values.
    - Binned DataFrames are automatically saved as CSV files with a '_bin{N}.csv' suffix.

    Parameters
    ----------
    file_path : str
        Path to the input CSV file containing the unbinned spectra.
    bin_sizes : list of int
        List of integers representing the desired bin sizes (number of points per bin).
    decimals : int, optional
        Number of decimal places for rounding the resulting values. Default is 5.

    Returns
    -------
    dict
        Dictionary where keys are the specific bin sizes and values are the corresponding 
        binned pandas DataFrames.
    """
    
    # --- Data Loading and Validation ---
    data = pd.read_csv(file_path, index_col=0)

    # Columns must be purely numeric (supports scientific notation)
    try:
        ppm_values = data.columns.to_numpy().astype(float)
    except Exception as e:
        raise ValueError(
            "Columns must be numeric (e.g., '-9e-05', '1.234', '2e-3'). "
            "The file contains non-numeric headers."
        ) from e

    # --- Initial Resolution Calculation ---
    if len(ppm_values) > 1:
        res = float(ppm_values[0]) - float(ppm_values[1])
        print('Initial resolution:', np.round_(np.abs(res), decimals=decimals))
    else:
        print('Initial resolution: N/A')

    # Convert data to numpy array
    data_np = data.to_numpy().astype(float)

    results = {}

    # --- Binning Process Loop ---
    for n in bin_sizes:
        
        # --- Bin Size Validation ---
        if n <= 0:
            raise ValueError(f"Bin size (n) must be > 0. Received: {n}")

        # --- Bin Allocation ---
        # Grouping by position (no interpolation), handling the last bin
        num_bins = len(ppm_values) // n
        if len(ppm_values) % n != 0:
            num_bins += 1  # Include the last bin even if incomplete

        # --- PPM Array Binning ---
        # Binning for ppm (mean of consecutive groups)
        ppm_binned = []
        i = 0
        while i < len(ppm_values):
            group = ppm_values[i:i + n]
            ppm_binned.append(np.mean(group))  # Average of values in the group
            i += n
        ppm_binned = np.round(ppm_binned, decimals=decimals)

        # --- Intensity Array Binning ---
        # Binning for intensities (mean of consecutive groups)
        intensities_binned = []
        for row in data_np:
            row_binned = []
            i = 0
            while i < len(row):
                group = row[i:i + n]
                row_binned.append(np.mean(group))  # Average of intensities in the group
                i += n
            intensities_binned.append(row_binned)

        intensities_binned = np.round(np.array(intensities_binned), decimals=decimals)

        # --- Column Renaming and DataFrame Assembly ---
        # Reconstruct column names, preserving specific suffixes if present (e.g., 'j' for imaginary parts)
        first_column_letter = re.findall(r'[a-zA-Z]', data.columns[0])
        suffix = first_column_letter[0] if first_column_letter else ""
        ppm_binned_names = [f"{val}{suffix}" for val in ppm_binned]

        # Create binned DataFrame
        df = pd.DataFrame(intensities_binned, index=data.index, columns=ppm_binned_names)

        # --- Export and Reporting ---
        # Save to file
        outfile = file_path.replace('.csv', f'_bin{n}.csv')
        df.to_csv(outfile)
        print(f"{outfile} saved.")

        # Final resolution
        if len(ppm_binned) > 1:
            res_final = ppm_binned[0] - ppm_binned[1]
            print('Final resolution:', np.round_(np.abs(res_final), decimals=decimals))
        else:
            print('Final resolution: N/A')

        results[n] = df

    return results

In [ ]:
###TPOT TRAIN

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from joblib import dump
from tpot import TPOTRegressor

def run_tpot_train(data_file, metadata_file, target_var='age', generations=3, population=50, offspring_size=100, random_seed=793, save_path='', use_std_scaler=True):
    """
    Runs the TPOT genetic algorithm to find an optimal regression model.

    Outputs generated:
    - Exports the best TPOT pipeline as a Python script: 'TPOT_{target_var}_{data_file_stem}.py'
    - Saves the fitted TPOT model as a binary file: 'modelTPOT_{target_var}_{data_file_stem}.bin'
    - If `use_std_scaler` is True, saves the fitted scaler: 'std_scalerTPOT_{target_var}_{data_file_stem}.bin'
    All files are saved in the directory specified by `save_path`.

    Parameters
    ----------
    data_file : str
        Filename of the input data CSV.
    metadata_file : str
        Filename of the input metadata CSV.
    target_var : str, optional
        Target variable to predict. Default is 'age'.
    generations : int, optional
        Number of generations for the TPOT genetic programming. Default is 3.
    population : int, optional
        Population size for TPOT. Default is 50.
    offspring_size : int, optional
        Offspring size for TPOT. Default is 100.
    random_seed : int, optional
        Random seed for reproducibility in train/test split. Default is 793.
    save_path : str, optional
        Path where the resulting models and scripts will be saved. Default is '' (current directory).
    use_std_scaler : bool, optional
        Whether to apply standard scaling (StandardScaler) to the data. Default is True.

    Returns
    -------
    tuple
        - df (pandas.DataFrame): Validation table with true ('Data') and predicted ('Pred') values.
        - R (float): Pearson correlation coefficient of the fitting.
        - mse (float): Mean Squared Error of the fitting.
    """

    # --- Data Loading & Validation ---
    print('Reading data...')
    data = pd.read_csv(data_file, index_col=0)
    # Ensure there are no duplicated sample indices to prevent data leakage or merging issues
    data = data[~data.index.duplicated(keep='first')]
    
    print('Reading metadata...')
    metadata = pd.read_csv(metadata_file, index_col=0, low_memory=False)

    print('Preprocessing...')

    # --- Metadata Alignment & Filtering ---
    # Filter only indices present in both data and metadata
    filtered_data = data.loc[data.index.intersection(metadata.index)]
    
    # Extract the target column from metadata
    md = metadata.loc[filtered_data.index, target_var]
    
    # Remove NaN values from target and their corresponding rows in data to ensure clean training
    valid_mask = ~md.isna()
    X = filtered_data[valid_mask]
    y = md[valid_mask]

    # --- Train/Test Split & Scaling ---
    # Split data into training and testing sets (80/20 split)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)
    
    if use_std_scaler:
        scaler = StandardScaler().fit(X_train)
        scaler_filename = f"{save_path}std_scalerTPOT_{target_var}_{Path(data_file).stem}.bin"
        dump(scaler, scaler_filename, compress=True)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    print(f'### {target_var} ###')
    print('Running TPOT...')
    
    # --- TPOT Genetic Optimization Phase ---
    tpot = TPOTRegressor(
        generations=generations, 
        population_size=population, 
        verbosity=2, 
        offspring_size=offspring_size,
        scoring='neg_mean_squared_error',
        cv=3
    )

    tpot.fit(X_train, y_train)
    
    # Score returns the negative mean squared error (as defined in scoring parameter)
    negmse = tpot.score(X_test, pd.DataFrame(y_test).values.ravel())
    pred = tpot.predict(X_test)
    
    # --- Exporting Models & Pipelines ---
    pipeline_filename = f"{save_path}TPOT_{target_var}_{Path(data_file).stem}.py"
    model_filename = f"{save_path}modelTPOT_{target_var}_{Path(data_file).stem}.bin"
    
    # Export the pure Python code of the best pipeline found by the genetic algorithm
    tpot.export(pipeline_filename)
    # Export the strictly fitted scikit-learn pipeline object for immediate predictive deployment
    dump(tpot.fitted_pipeline_, model_filename, compress=True)
    
    # --- Evaluation & Visualization ---
    score = np.corrcoef(y_test, pred)
    print(f'MSE: {-negmse} | R: {score[0, 1]}')
    
    plt.scatter(pd.DataFrame(y_test), pred)
    plt.xlabel('True Values')
    plt.ylabel('Predictions')
    plt.axis('equal')
    plt.axis('square')
    # Add diagonal line for perfect prediction reference (True == Predicted)
    _ = plt.plot([-1000, 1000], [-1000, 1000])
    plt.show()
    
    # Prepare the output DataFrame
    df = pd.DataFrame()
    df['Data'] = y_test
    df['Pred'] = pred
    R = score[0, 1]
    mse = -negmse
    
    return df, R, mse

In [ ]:
###RUN MODEL

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from joblib import dump

def run_model(model, data_file, metadata_file, target_var='age', skip_cv=False, k_folds=5, random_seed=793, save_path='', use_std_scaler=True):
    """
    Evaluates a machine learning model (e.g., TPOT pipeline) using optional cross-validation 
    and a final train/test validation approach.

    Outputs generated (saved in `save_path`):
    - 'OUTdataTPOT_{target_var}_CrossValidation.csv': Predictions vs True values for all CV folds.
    - 'OUTdataTPOT_{target_var}_finalTraining.csv': Predictions vs True values for the final training set.
    - 'OUTdataTPOT_{target_var}_finalValidation.csv': Predictions vs True values for the final validation set.
    - 'modelTPOT_{target_var}_{data_file_stem}.bin': The fitted model after the final training phase.
    - 'std_scalerTPOT_{target_var}_{data_file_stem}.bin': The fitted standard scaler (if `use_std_scaler` is True).

    Parameters
    ----------
    model : joblib pipeline or sklearn estimator
        Model to run and evaluate.
    data_file : str
        Filename of the input data CSV.
    metadata_file : str
        Filename of the input metadata CSV.
    target_var : str, optional
        Variable to run the model on. Default is 'age'.
    skip_cv : bool, optional
        Skip the K-Fold cross-validation phase if True. Default is False.
    k_folds : int, optional
        Number of folds for cross-validation. Default is 5.
    random_seed : int, optional
        Random seed for reproducibility. Default is 793.
    save_path : str, optional
        Directory path to save the output files. Default is '' (current directory).
    use_std_scaler : bool, optional
        Whether to apply standard scaling (StandardScaler) to the data. Default is True.

    Returns
    -------
    tuple
        - df (pandas.DataFrame): Final validation table with true and predicted values.
        - R (float): Pearson correlation coefficient of the final fitting.
        - mse (float): Mean Squared Error of the final fitting.
    """

    # --- Data Loading ---
    print('Reading data...')
    data_df = pd.read_csv(data_file, index_col=0)
    
    print('Reading metadata...')
    metadata_df = pd.read_csv(metadata_file, index_col=0, low_memory=False)
    
    print('Preprocessing...')

    # --- Preprocessing & Metadata Alignment ---
    # Remove duplicated indices in data to prevent data leakage
    data_df = data_df[~data_df.index.duplicated(keep='first')]
    
    # Merge data with metadata based on the index, keeping only the requested target variable
    merged_data = data_df.merge(metadata_df[[target_var]], left_index=True, right_index=True, how='inner')
    
    # Drop rows where the target variable is NaN to ensure a clean training matrix
    merged_data = merged_data.dropna(subset=[target_var])
    
    print(f'### {target_var} ###')
    
    # --- Global Train/Test Split ---
    # Split into general training and testing sets (80/20)
    X_train_f = merged_data.sample(frac=0.8, random_state=random_seed)
    X_test_f = merged_data.drop(X_train_f.index)
    
    y_train_f = X_train_f.pop(target_var)
    y_test_f = X_test_f.pop(target_var)

    exported_pipeline = model
    
    # --- Cross-Validation Phase ---
    if not skip_cv:
        kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_seed)
        
        print('Starting cross-validation...')
        print('')
        print('--------------------------------')

        cv_results_df = pd.DataFrame()
        
        # Rejoin features and labels for the CV splitting process
        data_cv = X_train_f.join(y_train_f, how='left')
        
        for fold, (train_ids, test_ids) in enumerate(kfold.split(data_cv)):
            print(f'FOLD {fold+1} of {k_folds}')
            print('--------------------------------')

            train_dataset_pre = data_cv.iloc[train_ids]
            test_dataset_pre = data_cv.iloc[test_ids]

            train_labels = train_dataset_pre.pop(target_var)
            test_labels = test_dataset_pre.pop(target_var)

            # CV fold scaling
            if use_std_scaler:
                scaler = StandardScaler().fit(train_dataset_pre.values)
                train_dataset = scaler.transform(train_dataset_pre.values)
                test_dataset = scaler.transform(test_dataset_pre.values)
            else:
                train_dataset = train_dataset_pre.values
                test_dataset = test_dataset_pre.values

            print('Train samples: ', train_dataset.shape[0], 'Test samples: ', test_dataset.shape[0], 'Points: ', train_dataset.shape[1])

            # CV fold fitting and prediction
            exported_pipeline.fit(train_dataset, train_labels)
            results = exported_pipeline.predict(test_dataset)

            mse = mean_squared_error(test_labels, results)
            score = np.corrcoef(test_labels, results)
            print('R=', score[0,1])
            print('MSE=', mse)
            
            plt.scatter(test_labels, results)
            plt.xlabel(f'Real {target_var}')
            plt.ylabel(f'Predicted {target_var}')
            plt.axis('equal')
            plt.axis('square')
            _ = plt.plot([-1000, 1000], [-1000, 1000])
            plt.show()

            # Append fold results for global CV evaluation
            fold_indices = test_labels.index.tolist()
            fold_true_data = test_labels.tolist()
            fold_pred_data = results.tolist()
            
            df_fold = pd.DataFrame(index=fold_indices)
            df_fold['Data'] = fold_true_data
            df_fold['Pred'] = fold_pred_data
            cv_results_df = pd.concat([cv_results_df, df_fold])

        # Global CV Results
        print('Cross-validation results:')
        print('R=', np.corrcoef(cv_results_df['Data'], cv_results_df['Pred'])[0,1])
        print('MSE=', mean_squared_error(cv_results_df['Data'], cv_results_df['Pred']))

        plt.scatter(cv_results_df['Data'], cv_results_df['Pred'], c='red')
        plt.xlabel(f'Real {target_var}')
        plt.ylabel(f'Predicted {target_var}')
        plt.axis('equal')
        plt.axis('square')
        _ = plt.plot([-1000, 1000], [-1000, 1000])
        plt.show()

        cv_label = f"OUTdataTPOT_{target_var}_CrossValidation.csv"
        cv_results_df.to_csv(f"{save_path}{cv_label}")

    # --- Final Validation Phase ---
    print('Starting the final validation...')
    
    # Final scaling on the complete training set
    if use_std_scaler:
        scaler = StandardScaler().fit(X_train_f.values)
        scaler_filename = f"{save_path}std_scalerTPOT_{target_var}_{Path(data_file).stem}.bin"
        dump(scaler, scaler_filename, compress=True)
        train_dataset = scaler.transform(X_train_f.values)
        test_dataset = scaler.transform(X_test_f.values)
    else:
        train_dataset = X_train_f.values
        test_dataset = X_test_f.values

    print('Running the final model...')
    print('Train samples: ', train_dataset.shape[0], 'Test samples: ', test_dataset.shape[0], 'Points: ', train_dataset.shape[1])
    
    exported_pipeline.fit(train_dataset, y_train_f)
    
    # Export the globally fitted model
    model_filename = f"{save_path}modelTPOT_{target_var}_{Path(data_file).stem}.bin"
    dump(exported_pipeline, model_filename, compress=True)
    
    results = exported_pipeline.predict(test_dataset)

    print('')
    print('Final validation results:')
    mse = mean_squared_error(y_test_f, results)
    score = np.corrcoef(y_test_f, results)
    final_r = score[0,1]
    final_mse = mse
    
    print('R=', final_r)
    print('MSE=', final_mse)
    
    plt.scatter(y_test_f, results, c='g')
    plt.xlabel(f'Real {target_var}')
    plt.ylabel(f'Predicted {target_var}')
    plt.axis('equal')
    plt.axis('square')
    _ = plt.plot([-1000, 1000], [-1000, 1000])
    plt.show()

    # --- Saving Training Results ---
    results_train = exported_pipeline.predict(train_dataset)
    train_indices = X_train_f.index.tolist()
    train_true_data = y_train_f.tolist()
    train_pred_data = results_train.tolist()
    
    df_train_out = pd.DataFrame(index=train_indices)
    df_train_out['Data'] = train_true_data
    df_train_out['Pred'] = train_pred_data
    train_label = f"OUTdataTPOT_{target_var}_finalTraining.csv"
    df_train_out.to_csv(f"{save_path}{train_label}")

    print('Training Metrics:')
    train_mse = mean_squared_error(y_train_f, results_train)
    train_score = np.corrcoef(y_train_f, results_train)
    print('R=', train_score[0,1])
    print('MSE=', train_mse)
    
    plt.scatter(y_train_f, results_train, c='blue')
    plt.xlabel(f'Real {target_var}')
    plt.ylabel(f'Predicted {target_var}')
    plt.axis('equal')
    plt.axis('square')
    _ = plt.plot([-1000, 1000], [-1000, 1000])
    plt.show()

    # --- Saving Validation Results ---
    test_indices = X_test_f.index.tolist()
    test_true_data = y_test_f.tolist()
    test_pred_data = results.tolist()
    
    df_test_out = pd.DataFrame(index=test_indices)
    df_test_out['Data'] = test_true_data
    df_test_out['Pred'] = test_pred_data
    test_label = f"OUTdataTPOT_{target_var}_finalValidation.csv"
    df_test_out.to_csv(f"{save_path}{test_label}")
    print('')

    return df_test_out, final_r, final_mse

In [ ]:
###APPLY MODEL TO NEW COHORT AND ANALYZE RESULTS

import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import torch
from joblib import load
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2_contingency, ks_2samp, ttest_ind

warnings.filterwarnings("ignore", category=UserWarning)

def extract_samples_to_match_stats(df1, column_df1, df2, column_df2, precision='high'):
    """
    For each value in df1, selects the closest unique value in df2 and creates a 
    DataFrame with the selected rows of df2.
    Stops or skips the comparison if the absolute difference between the compared 
    values exceeds 5.

    Parameters
    ----------
    df1 : pandas.DataFrame
        The first DataFrame (reference).
    column_df1 : str
        The column name from df1 to match against.
    df2 : pandas.DataFrame
        The second DataFrame (pool of available samples to select from).
    column_df2 : str
        The column name from df2 to search for the closest value.
    precision : str, optional
        Precision of mean/std matching. 
        'high' (default): breaks the loop if a match difference exceeds 5.
        'low' (or other): skips the current value (continues) if difference > 5.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the precisely selected samples from df2.
    """
    # --- Initialization ---
    # Create an empty DataFrame to store selected samples from df2
    selected_samples_df2 = pd.DataFrame(columns=df2.columns)

    # Use a set for fast lookup of already selected indices
    selected_indices_set = set()
    
    # Use a list to strictly preserve the order of the selected indices
    selected_indices_ordered = []

    # --- Sample matching loop ---
    # Iterate over each value in df1
    for value_df1 in df1[column_df1]:
        
        # Calculate the absolute differences between value_df1 and each value in df2
        differences = (df2[column_df2] - value_df1).abs()

        # Exclude indices that have already been selected
        available_indices = differences.index.difference(selected_indices_set)
        
        # Safety check: if no more samples are available, stop
        if len(available_indices) == 0:
            break

        # Find the index of the closest unique value in df2
        closest_index = differences.loc[available_indices].idxmin()

        # Threshold of 5 units (e.g., years). 
        # Prevents matching samples that are too biologically or statistically distant.
        # Check if the absolute difference between the compared values exceeds 5
        if differences.loc[closest_index] > 5:
            if precision == 'high':
                break
            else:
                continue

        # Add the selected row from df2 to selected_samples_df2
        selected_samples_df2.loc[len(selected_samples_df2)] = df2.loc[closest_index]

        # Add the selected index to our tracking variables
        selected_indices_set.add(closest_index)
        selected_indices_ordered.append(closest_index)

    # Assign the correctly ordered indices to the final DataFrame
    selected_samples_df2.index = selected_indices_ordered
    
    return selected_samples_df2

def analyze_age_new_sample_shap_stats(
    scaler_file, model_file, prev_data_file, new_data_file, new_metadata_file, prev_input_data_file,
    model_type='pytorch', sampling_precision='high', save_fig=False, save_csv=False, save_path='', sample_name='control'
):
    """
    Analyzes a new sample against an age model and adds:
      - Global SHAP analysis for the reference and the new sample.
      - Feature importance differences (new - reference) sorted and plotted.
      - Standardized original variables comparison (z-score relative to reference),
        with a bar plot of Δ means (new - ref) ordered by feature.
      - Custom titles for SHAP plots and previous prints.

    Requirements:
      - prev_input_data_file: CSV containing the FEATURES of the reference set (same columns as new_data_file).
      - 'extract_samples_to_match_stats' function must be defined in the environment.

    SHAP Behavior:
      - Uses available samples for each group; if >100, caps at 100.
      - Background is taken from the reference with the same 100 sample cap.
    """

    # --- Internal utilities ---
    def _pick_shap_explainer(fitted_model, m_type, bg_data):
        """
        Robust explainer:
          - TPOT/sklearn: forces KernelExplainer on model.predict
          - PyTorch: predictive function + KernelExplainer
        Important: bg_data must be an ndarray.
        """
        if not isinstance(bg_data, np.ndarray):
            bg_data = np.asarray(bg_data)

        if m_type == 'TPOT':
            f = fitted_model.predict  # Stable callable for pipelines
            return shap.KernelExplainer(f, bg_data)

        elif m_type == 'pytorch':
            fitted_model.eval()
            def _predict_fn(x):
                with torch.no_grad():
                    t = torch.tensor(x, dtype=torch.float32)
                    return fitted_model(t).cpu().numpy().ravel()
            return shap.KernelExplainer(_predict_fn, bg_data)

        else:
            raise ValueError("model_type must be 'pytorch' or 'TPOT'.")

    def _cap(x_data, cap_size=100, seed=42):
        """
        WARNING: SHAP KernelExplainer is extremely computationally expensive.
        Returns all rows if len(X) <= cap; otherwise, a random sample of size 'cap' (default 100).
        """
        if len(x_data) <= cap_size:
            return x_data
        rs = np.random.RandomState(seed)
        idx = rs.choice(len(x_data), size=cap_size, replace=False)
        return x_data[idx]

    # --- Reading and preprocessing ---
    print('Reading data...')
    # Previous model output (reference with 'Age' and 'Metabolic age')
    ref_df = pd.read_csv(prev_data_file, index_col=0)
    ref_df = ref_df.rename(columns={'Data': 'Age', 'Pred': 'Metabolic age'})

    # Reference features (for SHAP) and new features
    x_ref_full_df = pd.read_csv(prev_input_data_file, index_col=0)
    new_data_df = pd.read_csv(new_data_file, index_col=0)
    new_metadata_df = pd.read_csv(new_metadata_file, index_col=0)

    # Standardize all indices to clean text to prevent mismatch blocks
    for dataframe in [ref_df, x_ref_full_df, new_data_df, new_metadata_df]:
        dataframe.index = dataframe.index.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

    # Align reference: common indices with ref_df
    common_idx = x_ref_full_df.index.intersection(ref_df.index)

    if len(common_idx) == 0:
        raise ValueError("No common indices between prev_input_data_file and prev_data_file.")
    x_ref_full_df = x_ref_full_df.loc[common_idx]
    ref_df = ref_df.loc[common_idx]

    # Verify/sort columns so reference and new data match
    if list(x_ref_full_df.columns) != list(new_data_df.columns):
        if set(x_ref_full_df.columns) == set(new_data_df.columns):
            new_data_df = new_data_df[x_ref_full_df.columns]
        else:
            raise ValueError("Columns of prev_input_data_file and new_data_file do not match.")

    # Dump actual ages of the new sample and clean missing/NaN values
    keep_idx = []
    md_pc = []
    for i in new_data_df.index:
        if i in new_metadata_df.index:
            val = new_metadata_df.loc[i, 'age']
            if not math.isnan(val):
                keep_idx.append(i)
                md_pc.append(val)
    new_data_df = new_data_df.loc[keep_idx]
    md_pc = [new_metadata_df.loc[i, 'age'] for i in new_data_df.index]

    # Scaling for the model
    sc = load(scaler_file)
    x_ref_scaled = sc.transform(x_ref_full_df.values)
    x_new_scaled = sc.transform(new_data_df.values)

    # --- Predictions ---
    if model_type == 'pytorch':
        model_pt = torch.load(model_file)
        model_pt.eval()
        input_pc = torch.tensor(x_new_scaled).float()
        pred_pc = list(model_pt(input_pc).detach().numpy().ravel())
        fitted_model = model_pt
    elif model_type == 'TPOT':
        exported_pipeline = load(model_file)
        pred_pc = exported_pipeline.predict(x_new_scaled)
        fitted_model = exported_pipeline
    else:
        print('Wrong model type.')
        return

    print('\nPlot of predicted ages of new data and the full reference data')
    plt.scatter(ref_df['Age'], ref_df['Metabolic age'])
    plt.xlabel('Age')
    plt.ylabel('Metabolic age')
    plt.axis('equal')
    plt.axis('square')
    _ = plt.plot([-1000, 1000], [-1000, 1000])
    plt.scatter(md_pc, pred_pc)
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_plot.svg", bbox_inches='tight')
    plt.show()

    # DataFrame of new predictions
    new_pred_df = pd.DataFrame(index=new_data_df.index)
    new_pred_df['Age'] = list(md_pc)
    new_pred_df['Metabolic age'] = list(pred_pc)

    # --- Sampling / matching statistics ---
    print('SAMPLING')
    print(f"Reference data age: {ref_df['Age'].mean()} +- {ref_df['Age'].std()} ( {ref_df['Age'].count()} samples )")
    print(f"New sample data age: {new_pred_df['Age'].mean()} +- {new_pred_df['Age'].std()} ( {new_pred_df['Age'].count()} samples )")

    # Uses the previously defined extraction function to balance the cohorts
    if sampling_precision == 'high':
        ref_df_sampled = extract_samples_to_match_stats(new_pred_df, 'Age', ref_df, 'Age')
    else:
        ref_df_sampled = extract_samples_to_match_stats(new_pred_df, 'Age', ref_df, 'Age', precision='low')

    print(f"\nReference data age after sampling: {ref_df_sampled['Age'].mean()} +- {ref_df_sampled['Age'].std()} ( {ref_df_sampled['Age'].count()} samples )")
    print(f"Reference data metabolic age after sampling: {ref_df_sampled['Metabolic age'].mean()} +- {ref_df_sampled['Metabolic age'].std()} ( {ref_df_sampled['Metabolic age'].count()} samples )")
    print(f"New sample data metabolic age: {new_pred_df['Metabolic age'].mean()} +- {new_pred_df['Metabolic age'].std()} ( {new_pred_df['Metabolic age'].count()} samples )\n")

    ref_df_sampled.to_csv(new_data_file.replace('.csv', '_Sampling.csv'))
    new_pred_df.to_csv(new_data_file.replace('.csv', '_SamplingDis.csv'))

    # --- Regression and categories (Metabolic banding) ---
    lm = LinearRegression().fit(ref_df_sampled[['Age']], ref_df_sampled[['Metabolic age']])

    def _categorize(dataset):
        calc_cat = []
        m_dist = []
        for i in dataset.index:
            a = dataset.loc[i, 'Age']
            ma = dataset.loc[i, 'Metabolic age']
            
            # Calculate residual (distortion)
            m_dist.append((ma - (lm.coef_ * a + lm.intercept_))[0][0])
            
            # ±5 years tolerance band to define categorical metabolic profiles
            ma_calc = lm.coef_ * a + lm.intercept_ + 5
            ma_calc2 = ma_calc - 10
            
            if ma > ma_calc:
                calc_cat.append('met_older')
            elif ma < ma_calc2:
                calc_cat.append('met_younger')
            else:
                calc_cat.append('met_normal')
                
        dataset['Metabolic age category'] = calc_cat
        dataset['Metabolic distortion'] = m_dist
        return dataset

    ref_df_sampled = _categorize(ref_df_sampled)
    new_pred_df = _categorize(new_pred_df)

    # --- Plots and statistical testing ---
    print('Plot of predicted ages of new samples colored by metabolic age category and the reference (black)')
    sns.scatterplot(data=ref_df_sampled, x='Age', y='Metabolic age', alpha=0.5, color='black')
    sns.scatterplot(data=new_pred_df, x='Age', y='Metabolic age', hue='Metabolic age category', alpha=0.5, palette='coolwarm')
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_plot2.svg", bbox_inches='tight')
    
    plt.savefig('000NegControl_plot2.svg', bbox_inches='tight')
    plt.show()

    print('Predicted ages for each metabolic age category of the reference')
    print(ref_df_sampled.groupby('Metabolic age category')['Metabolic age'].agg(['mean', 'std', 'count']))
    print('\nPredicted ages for each metabolic age category of the new sample')
    print(new_pred_df.groupby('Metabolic age category')['Metabolic age'].agg(['mean', 'std', 'count']))
    print('')

    met_type = pd.api.types.CategoricalDtype(categories=['met_normal', 'met_older', 'met_younger'], ordered=False)
    ref_df_sampled['Metabolic age category'] = ref_df_sampled['Metabolic age category'].astype(met_type)
    new_pred_df['Metabolic age category'] = new_pred_df['Metabolic age category'].astype(met_type)

    c1 = list(ref_df_sampled.groupby('Metabolic age category')['Metabolic age'].agg('count'))
    c2 = list(new_pred_df.groupby('Metabolic age category')['Metabolic age'].agg('count'))
    marks = ['met_normal', 'met_older', 'met_younger']

    print('Distribution of metabolic age categories in the reference:')
    plt.pie(c1, labels=marks)
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_pieRef.svg", bbox_inches='tight')
    plt.show()

    print('Distribution of metabolic age categories in the new sample:')
    plt.pie(c2, labels=marks)
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_pie2.svg", bbox_inches='tight')
    plt.show()

    chi_data = [c1, c2]
    print('\nChi-square analysis for metabolic age category distributions:')
    stat, p, dof, expected = chi2_contingency(chi_data)
    print(f"p-value (lower confirms that samples are different) is {p}")

    print('\nDistribution of metabolic distortion of reference (blue) and new sample (orange):')
    fig, ax = plt.subplots()
    sns.distplot(ref_df_sampled['Metabolic distortion'], bins=range(-50, 50, 5), ax=ax, kde=False, norm_hist=True)
    sns.distplot(new_pred_df['Metabolic distortion'], bins=range(-50, 50, 5), ax=ax, kde=False, norm_hist=True)
    ax.set_xlim([-50, 50])
    ax.set_ylabel('')
    ax.set_yticks([])
    ax.set_yticklabels([])
    ax.set_xticks(ax.get_xticks())
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_metDistortion.svg", bbox_inches='tight')
    plt.show()

    if save_csv: 
        new_pred_df.to_csv(f"{save_path}{sample_name}_predictions.csv")
        
    print(f"Reference data metabolic age distortion after sampling: {ref_df_sampled['Metabolic distortion'].mean()} +- {ref_df_sampled['Metabolic distortion'].std()} ( {ref_df_sampled['Metabolic distortion'].count()} samples )")
    print(f"New sample data metabolic age distortion: {new_pred_df['Metabolic distortion'].mean()} +- {new_pred_df['Metabolic distortion'].std()} ( {new_pred_df['Metabolic distortion'].count()} samples )")
    
    print('\nKolmogorov-Smirnov test analysis for metabolic age distortion distributions:')
    p_ks = ks_2samp(ref_df_sampled['Metabolic distortion'], new_pred_df['Metabolic distortion'])[1]
    print(f"p-value (lower confirms that samples are different) is {p_ks}")

    print('\nT-test analysis for metabolic age distortion means:')
    # equal_var=False applies Welch's t-test (more robust if variances are unequal)
    t_stat, p_t = ttest_ind(ref_df_sampled['Metabolic distortion'], new_pred_df['Metabolic distortion'], equal_var=False)
    print(f"p-value (lower confirms that means are different) is {p_t}")

    # --- SHAP analysis ---
    print('\n=== SHAP analysis (global importances & differences) ===')

    # Reference indices used in sampling (to align with x_ref_full_df)
    ref_idx_for_shap = [i for i in ref_df_sampled.index if i in x_ref_full_df.index]
    # If it becomes empty, use the whole reference
    x_ref_for_shap_full = x_ref_full_df.loc[ref_idx_for_shap] if len(ref_idx_for_shap) else x_ref_full_df

    # Rescaling subsets for the model
    x_ref_for_shap_scaled = sc.transform(x_ref_for_shap_full.values)
    x_new_for_shap_scaled = x_new_scaled

    # Applying the 100-sample cap for computational feasibility
    x_bg = _cap(x_ref_scaled, cap_size=100)
    x_ref_shap = _cap(x_ref_for_shap_scaled, cap_size=100)
    x_new_shap = _cap(x_new_for_shap_scaled, cap_size=100)

    feature_names = list(x_ref_full_df.columns)
    explainer = _pick_shap_explainer(fitted_model, model_type, x_bg)

    # KernelExplainer uses .shap_values; nsamples="auto"
    shap_ref_vals = explainer.shap_values(x_ref_shap, nsamples="auto")
    shap_new_vals = explainer.shap_values(x_new_shap, nsamples="auto")
    
    imp_ref = np.abs(shap_ref_vals).mean(axis=0)
    imp_new = np.abs(shap_new_vals).mean(axis=0)
    delta = imp_new - imp_ref
    order = np.argsort(-np.abs(delta))

    imp_df = pd.DataFrame({
        'feature': np.array(feature_names),
        '|SHAP|_ref': imp_ref,
        '|SHAP|_new': imp_new,
        'delta_new_minus_ref': delta
    }).iloc[order].reset_index(drop=True)

    # Plot differences (top 10)
    top_k = 10
    print('Figure: Δ |SHAP| (sample − reference) by feature (top changes)')
    
    plot_width = 5.0
    plot_height = max(4.0, top_k * 0.35)
    
    left_margin = 3.5  
    right_margin = 0.5
    top_margin = 0.5
    bottom_margin = 1.0
    
    fig_width = left_margin + plot_width + right_margin
    fig_height = bottom_margin + plot_height + top_margin
    
    fig = plt.figure(figsize=(fig_width, fig_height))
    ax = fig.add_axes([
        left_margin / fig_width,      
        bottom_margin / fig_height,   
        plot_width / fig_width,       
        plot_height / fig_height      
    ])

    sns.barplot(
        y=imp_df['feature'].head(top_k),
        x=imp_df['delta_new_minus_ref'].head(top_k),
        ax=ax, orient='h'
    )
    ax.axvline(0, lw=1, ls='--', color='k')
    ax.set_xlabel('Δ |SHAP| (sample − reference)')
    ax.set_ylabel('Features')
    ax.set_title('SHAP changes between groups')
    
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_shap_delta_bar.svg") 
    plt.show()
    
    top_10_features_ref = np.argsort(np.abs(shap_ref_vals).mean(axis=0))[-10:]
    top_10_features_new = np.argsort(np.abs(shap_new_vals).mean(axis=0))[-10:]
    
    # Summary plots per group
    try:
        plot_height_shap = max(4.0, 10 * 0.35)
        fig_height_shap = bottom_margin + plot_height_shap + top_margin
        
        print('Figure: SHAP summary — Reference')
        fig_ref = plt.figure(figsize=(fig_width, fig_height_shap))
        ax_ref = fig_ref.add_axes([
            left_margin / fig_width,
            bottom_margin / fig_height_shap,
            plot_width / fig_width,
            plot_height_shap / fig_height_shap
        ])
        
        shap.summary_plot(
            shap_ref_vals[:, top_10_features_ref], 
            features=x_ref_shap[:, top_10_features_ref], 
            feature_names=np.array(feature_names)[top_10_features_ref], 
            show=False,
            plot_size=None 
        )
        if save_fig: 
            plt.savefig(f"{save_path}{sample_name}_shap_summary_ref.svg")
        plt.show()

        print('Figure: SHAP summary — New sample')
        fig_new = plt.figure(figsize=(fig_width, fig_height_shap))
        ax_new = fig_new.add_axes([
            left_margin / fig_width,
            bottom_margin / fig_height_shap,
            plot_width / fig_width,
            plot_height_shap / fig_height_shap
        ])
        
        shap.summary_plot(
            shap_new_vals[:, top_10_features_new], 
            features=x_new_shap[:, top_10_features_new], 
            feature_names=np.array(feature_names)[top_10_features_new], 
            show=False,
            plot_size=None
        )
        if save_fig: 
            plt.savefig(f"{save_path}{sample_name}_shap_summary_new.svg")
        plt.show()
        
    except Exception as e:
        print("Warning: Could not generate SHAP summary plots:", e)

    if save_csv:
        out_csv = f"{save_path}{sample_name}_shap_importances.csv"
        imp_df.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

    # --- Original variables analysis (z-score) ---
    print('\n=== Standardized original variables analysis (z-score) ===')
    
    x_ref_vars_full = x_ref_for_shap_full.values
    x_new_vars_full = new_data_df.values

    # Apply the same 100-sample cap for consistency with SHAP arrays
    x_ref_vars = _cap(x_ref_vars_full, cap_size=100)
    x_new_vars = _cap(x_new_vars_full, cap_size=100)

    # Standardize with reference group statistics
    z_scaler = StandardScaler().fit(x_ref_vars)
    z_ref = z_scaler.transform(x_ref_vars)
    z_new = z_scaler.transform(x_new_vars)

    z_mean_ref = z_ref.mean(axis=0)
    z_mean_new = z_new.mean(axis=0)

    z_delta = z_mean_new - z_mean_ref
    z_order = np.argsort(-np.abs(z_delta))

    vars_df = pd.DataFrame({
        'feature': np.array(feature_names),
        'zmean_ref': z_mean_ref,
        'zmean_new': z_mean_new,
        'delta_new_minus_ref': z_delta
    }).iloc[z_order].reset_index(drop=True)

    top_k_vars = min(25, len(feature_names))
    print('Figure: Δ mean (z-score) of original variables (new − reference), top changes')
    fig, ax = plt.subplots(figsize=(8, max(4, top_k_vars * 0.35)))
    sns.barplot(
        y=vars_df['feature'].head(top_k_vars),
        x=vars_df['delta_new_minus_ref'].head(top_k_vars),
        ax=ax, orient='h'
    )
    ax.axvline(0, lw=1, ls='--', color='k')
    ax.set_xlabel('Δ mean in z-score (new − reference)')
    ax.set_ylabel('Feature (top changes)')
    ax.set_title('Change in standardized original variables')
    plt.tight_layout()
    
    if save_fig: 
        plt.savefig(f"{save_path}{sample_name}_vars_delta_bar.svg", bbox_inches='tight')
    plt.show()

    if save_csv:
        out_csv2 = f"{save_path}{sample_name}_vars_zscore_deltas.csv"
        vars_df.to_csv(out_csv2, index=False)
        print(f"Saved: {out_csv2}")

    # --- Feature extraction and final outputs ---
    # Sort features by absolute SHAP value (reference sample)
    top_shap_features_ref = imp_df[['feature', '|SHAP|_ref']].sort_values(by='|SHAP|_ref', ascending=False).head(10).reset_index(drop=True)
    print(f'\nReference - Top 10 features with highest mean absolute SHAP (reference sample):')
    print(top_shap_features_ref)

    # Sort features by absolute SHAP value (new sample)
    top_shap_features_new = imp_df[['feature', '|SHAP|_new']].sort_values(by='|SHAP|_new', ascending=False).head(10).reset_index(drop=True)
    print(f'\n{sample_name} - Top 10 features with highest mean absolute SHAP (new sample):')
    print(top_shap_features_new)

    return {
        'ref_predictions': ref_df_sampled,
        'new_predictions': new_pred_df,
        'shap_importances': imp_df,
        'vars_zscore_deltas': vars_df
    }